# Macro/News Agent + Risk Rules (Gonçalo)

**What this notebook does**
1. **Risk rules** – `check_risk()` checks the Portfolio Manager's proposed orders against the competition rules and our safety rules (**max 20% per ETF**, **cap on leveraged ETFs**).
2. **Macro/News** – downloads recent headlines on **interest rates, geopolitics and commodities** and has the AI (Gemini) summarise them and say what they mean for our ETF themes.
3. **LangGraph** – connects both in a mini-agent: Macro/News → (Portfolio Manager placeholder) → Risk.

**How to read it**
- Lines ending with `# <<< ADAPT` are the generic parts to change.
- Parts 1–4 work **without** the AI. The AI summary (5) needs your own Gemini key in a `.env` file (never on GitHub).
- Run the cells **from top to bottom** (each cell uses things created above).
- Uses Morgane's cleaned `data/prices.csv` (columns `code`, `theme`, `close`, `adj_close`, `leverage`).

## 0. Install (once)

In [ ]:
%pip install -q pandas feedparser langgraph langchain-openai python-dotenv

## 1. ✏️ Settings

In [ ]:
import os, json
from pathlib import Path
import pandas as pd
import feedparser
from urllib.parse import quote

# Find the project folder automatically (the one with data/prices.csv), so it works on everyone's PC
here = Path.cwd()
for folder in [here, *here.parents]:
    if (folder / "data" / "prices.csv").exists():
        os.chdir(folder)
        break
print("Project folder:", Path.cwd())
PRICES_FILE    = Path("data/prices.csv")
PORTFOLIO_FILE = Path("trading/portfolio.csv")   # created later by the team; until then: all cash

# ---- Competition rules (FAQ) - do not change ----
MAX_ORDERS_PER_DAY = 5        # including sells
MIN_CASH_EUR       = 0.0      # no borrowing

# ---- Our safety rules (team decision) ----
MAX_WEIGHT_PER_ETF   = 0.20   # <<< ADAPT  max 20% of the portfolio in one ETF
MAX_LEVERAGED_WEIGHT = 0.15   # <<< ADAPT  max 15% in all leveraged/inverse ETFs together
MIN_ORDER_EUR        = 500.0  # <<< ADAPT  ignore tiny orders (we only have 5 per day)
STARTING_CAPITAL     = 100_000.0   # <<< ADAPT  check the official amount

## 2. Load data

- **ETF list** (code, theme, leverage) comes straight from `prices.csv`.
- **T-1 price** = last `close` per ETF (orders are executed at the previous day's close).
- **Portfolio**: if `trading/portfolio.csv` doesn't exist yet, we start with 100% cash.

In [ ]:
prices = pd.read_csv(PRICES_FILE, parse_dates=["date"])
universe = prices.groupby("code").agg(theme=("theme", "last"), leverage=("leverage", "last")).reset_index()
last_close = prices.sort_values("date").groupby("code")["close"].last()

def load_portfolio():
    if PORTFOLIO_FILE.exists():
        return pd.read_csv(PORTFOLIO_FILE)          # columns: code, units  (+ a CASH row)
    return pd.DataFrame({"code": ["CASH"], "units": [STARTING_CAPITAL]})

portfolio = load_portfolio()
print(f"{len(universe)} ETFs, prices up to {prices['date'].max():%Y-%m-%d}")
print("Leveraged/inverse:", ", ".join(universe.loc[universe["leverage"] != 1, "code"]))
portfolio

## 3. Risk rules – `check_risk()`

**Order format** (to agree with Francesco):
```python
[{"code": "NASDAQ", "action": "buy", "amount_eur": 10000},
 {"code": "GOLD",   "action": "sell", "amount_eur": 5000}]
```
**Rules, in order**
1. Max **5 orders** per day (incl. sells) – extra ones rejected
2. ETF must be in our list, action buy/sell, amount ≥ minimum
3. Can't sell more than we hold
4. **Sells first** (they free cash), then buys in the proposed order, at the **T-1 close**
5. **Max 20% per ETF** after the trade → buy reduced to fit
6. **Cap on leveraged ETFs** (all together) → buy reduced to fit
7. **No borrowing** → if cash runs out, the last buys are cut (FAQ rule)

In [ ]:
def check_risk_rules(orders, portfolio=None):
    portfolio = load_portfolio() if portfolio is None else portfolio
    px = last_close
    lev = universe.set_index("code")["leverage"].to_dict()
    holdings = portfolio[portfolio["code"] != "CASH"].set_index("code")["units"].to_dict()
    cash = float(portfolio.loc[portfolio["code"] == "CASH", "units"].sum())
    value = lambda c: holdings.get(c, 0) * px.get(c, 0)
    total = cash + sum(value(c) for c in holdings)          # portfolio value before trading
    approved, rejected = [], []

    # Rule 1: max 5 orders
    for o in orders[MAX_ORDERS_PER_DAY:]:
        rejected.append({**o, "reason": f"more than {MAX_ORDERS_PER_DAY} orders"})
    orders = orders[:MAX_ORDERS_PER_DAY]

    # Rule 2: basic checks
    valid = []
    for o in orders:
        c, a, amt = o.get("code"), str(o.get("action", "")).lower(), float(o.get("amount_eur", 0))
        if c not in lev:
            rejected.append({**o, "reason": "ETF not in our list"})
        elif a not in ("buy", "sell"):
            rejected.append({**o, "reason": "action must be buy or sell"})
        elif amt < MIN_ORDER_EUR:
            rejected.append({**o, "reason": f"below {MIN_ORDER_EUR:.0f} EUR"})
        else:
            valid.append({"code": c, "action": a, "amount_eur": amt})

    # Rules 3-4: sells first
    for o in [o for o in valid if o["action"] == "sell"]:
        held = value(o["code"])
        if held <= 0:
            rejected.append({**o, "reason": "we don't hold it"}); continue
        amt = min(o["amount_eur"], held)
        holdings[o["code"]] -= amt / px[o["code"]]
        cash += amt
        approved.append({**o, "amount_eur": round(amt, 2), "units": round(amt / px[o["code"]], 4),
                         "price_T-1": round(px[o["code"]], 4),
                         "note": "cut to what we hold" if amt < o["amount_eur"] else ""})

    # Rules 5-7: buys in proposed order
    lev_value = sum(value(c) for c in holdings if lev.get(c, 1) != 1)
    for o in [o for o in valid if o["action"] == "buy"]:
        c, amt, why = o["code"], o["amount_eur"], []
        room = MAX_WEIGHT_PER_ETF * total - value(c)           # Rule 5: max 20% per ETF
        if amt > room:
            amt = max(room, 0)
            why.append(f"max {MAX_WEIGHT_PER_ETF:.0%} per ETF")
        if lev.get(c, 1) != 1:                                  # Rule 6: cap on leveraged ETFs
            room_lev = MAX_LEVERAGED_WEIGHT * total - lev_value
            if amt > room_lev:
                amt = max(room_lev, 0)
                why.append(f"max {MAX_LEVERAGED_WEIGHT:.0%} in leveraged ETFs")
        if amt > cash - MIN_CASH_EUR:                           # Rule 7: no borrowing
            amt = max(cash - MIN_CASH_EUR, 0)
            why.append("not enough cash")
        if amt < MIN_ORDER_EUR:
            rejected.append({**o, "reason": "; ".join(why) or "too small"}); continue
        holdings[c] = holdings.get(c, 0) + amt / px[c]
        cash -= amt
        if lev.get(c, 1) != 1: lev_value += amt
        approved.append({**o, "amount_eur": round(amt, 2), "units": round(amt / px[c], 4),
                         "price_T-1": round(px[c], 4), "note": "; ".join(why)})

    weights = {c: float(round(value(c) / total, 3)) for c in holdings if holdings[c] > 1e-9}
    weights["CASH"] = float(round(cash / total, 3))
    return {"approved": approved, "rejected": rejected,
            "cash_after_eur": round(cash, 2), "weights_after": weights}

### 3b. Test with example orders (designed to break the rules)

In [ ]:
test_orders = [
    {"code": "NASDAQ",     "action": "buy",  "amount_eur": 30000},   # > 20% -> cut to 20,000
    {"code": "NASDAQ 3X",  "action": "buy",  "amount_eur": 25000},   # leveraged -> cut to 15,000
    {"code": "COPPER LVG", "action": "buy",  "amount_eur": 5000},    # leveraged cap already used -> rejected
    {"code": "GOLD",       "action": "sell", "amount_eur": 1000},    # we don't hold it -> rejected
    {"code": "XYZ",        "action": "buy",  "amount_eur": 1000},    # not in our list -> rejected
    {"code": "WORLD",      "action": "buy",  "amount_eur": 10000},   # 6th order -> rejected
]
print(json.dumps(check_risk_rules(test_orders), indent=2))

## 4. ✏️ Macro/News – headlines per topic

Free source: **Google News RSS** (last 7 days). Change the topics/search words if needed.

In [ ]:
NEWS_TOPICS = {                                                              # <<< ADAPT
    "interest_rates": "ECB OR Federal Reserve interest rates",
    "ecb_europe":     "ECB Lagarde euro zone",
    "geopolitics":    "geopolitical risk markets",
    "commodities":    "oil OR copper OR gold prices",
}
N_HEADLINES = 8                                                              # <<< ADAPT

def get_headlines(query, n=N_HEADLINES):
    url = f"https://news.google.com/rss/search?q={quote(query)}+when:7d&hl=en&gl=US&ceid=US:en"
    try:
        entries = feedparser.parse(url).entries[:n]
        return [f"- {e.title} ({e.get('published', '')[:16]})" for e in entries] or ["- no headlines found"]
    except Exception as e:
        return [f"- news unavailable ({type(e).__name__})"]

for topic, query in NEWS_TOPICS.items():
    print(f"\n== {topic} ==")
    print("\n".join(get_headlines(query)))

## 5a. Gemini settings

The API key is read from a **`.env`** file in the project folder (copy `.env.example` to `.env` and put your own key there).
`.env` is in `.gitignore`, so the key **never goes to GitHub**. Never write the key in this notebook.

```
LLM_API_KEY=your-key-here
```

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                                   # reads LLM_API_KEY from the .env file
os.environ.setdefault("LLM_BASE_URL", "https://generativelanguage.googleapis.com/v1beta/openai/")

# Models to try, in order (the first ones worked in our tests). Stops at the first that answers.
CANDIDATE_MODELS = ["gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.5-flash",
                    "gemini-3.5-flash-lite", "gemini-flash-latest"]           # <<< ADAPT if needed

if not os.environ.get("LLM_API_KEY"):
    print("No LLM_API_KEY found - create a .env file (see above). The rest still runs without AI.")
else:
    client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"],
                    timeout=20, max_retries=0)
    os.environ["LLM_MODEL"] = ""
    for m in CANDIDATE_MODELS:
        try:
            client.chat.completions.create(model=m, messages=[{"role": "user", "content": "Say hi"}])
            os.environ["LLM_MODEL"] = m
            break
        except Exception as e:
            print(f"{m}: not available ({type(e).__name__})")
    print("Gemini ready:", os.environ["LLM_MODEL"] or "NO MODEL AVAILABLE - try again later")

## 5b. Macro/News – AI summary (one request for all topics)

The **code** collects the headlines; the **AI** only summarises them and links them to our ETF themes.
One single request for all topics → faster and stays within the free-tier limit.
Returns `{"summary": AI text, "headlines": raw headlines per topic}` (headlines kept so every conclusion can be checked).

In [ ]:
import time

THEMES = sorted(universe["theme"].unique())

def get_llm():
    """Connects to Gemini. Returns None if the settings are missing."""
    if not all(os.environ.get(k) for k in ("LLM_BASE_URL", "LLM_MODEL", "LLM_API_KEY")):
        return None
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"],
                      model=os.environ["LLM_MODEL"], temperature=0, timeout=90, max_retries=1)

def get_macro_news(retries=2, pause=20):
    """1) the code collects the headlines of every topic  2) the AI summarises them in ONE request."""
    news = {topic: get_headlines(query) for topic, query in NEWS_TOPICS.items()}
    news_text = "\n\n".join(f"[{topic}]\n" + "\n".join(h) for topic, h in news.items())

    llm = get_llm()
    if llm is None:
        return {"summary": "[AI not configured]", "headlines": news}

    prompt = f"""You are the macro analyst of a team managing an ETF portfolio for the next 2 weeks.

Headlines from the last 7 days, grouped by topic:
{news_text}

Our ETF themes: {", ".join(THEMES)}

Rules:
- Use ONLY the headlines above. Do not invent numbers, dates or facts.
- Ignore headlines that are not relevant for financial markets.
- Our ETFs trade in EUR and many are European: consider the ECB, not only the Fed.
- If topics point in different directions for a theme, give one net view and explain the trade-off.
- Be concise.

For EACH topic ({", ".join(NEWS_TOPICS)}) write:
1. Key developments: max 3 short bullet points.
2. Affected ETF themes: theme -> positive/negative, with a one-line reason.
3. Tone: positive, neutral or negative.

Finish with an OVERALL VIEW (2-3 sentences): what this means for our portfolio in the next 2 weeks
(which themes look favourable, which look risky)."""                       # <<< ADAPT prompt

    for attempt in range(retries + 1):
        try:
            return {"summary": llm.invoke(prompt).content, "headlines": news}
        except Exception as e:
            print(f"AI busy ({type(e).__name__}), waiting {pause}s and trying again...", flush=True)
            time.sleep(pause)
    return {"summary": "[AI unavailable - see raw headlines]", "headlines": news}

macro_view = get_macro_news()
print(macro_view["summary"])

## 6. Mini-agent with LangGraph

Macro/News → Portfolio Manager (**placeholder** until Francesco's part is ready) → Risk.
The Portfolio Manager will read `state["macro"]["summary"]` and return orders in the agreed format.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

class State(TypedDict):
    macro: dict      # AI summary + raw headlines (Macro/News agent)
    orders: list     # proposed orders (Portfolio Manager)
    risk: dict       # checked orders (Risk agent)

def macro_node(state):
    print("1) Macro/News: collecting headlines and asking the AI...", flush=True)
    return {"macro": get_macro_news()}

def portfolio_node(state):
    print("2) Portfolio Manager: proposing orders (placeholder)...", flush=True)
    # PLACEHOLDER - Francesco's Portfolio Manager will replace this (it will use state["macro"])  # <<< ADAPT
    return {"orders": [{"code": "WORLD", "action": "buy", "amount_eur": 25000},
                       {"code": "NASDAQ 3X", "action": "buy", "amount_eur": 20000},
                       {"code": "OVERNIGHT CASH", "action": "buy", "amount_eur": 10000}]}

def risk_node(state):
    print("3) Risk: checking the orders against our rules...", flush=True)
    return {"risk": check_risk_rules(state["orders"])}

g = StateGraph(State)
g.add_node("macro", macro_node)
g.add_node("portfolio", portfolio_node)
g.add_node("risk", risk_node)
g.add_edge(START, "macro")
g.add_edge("macro", "portfolio")
g.add_edge("portfolio", "risk")
g.add_edge("risk", END)
app = g.compile()

try:
    display(Image(app.get_graph().draw_mermaid_png()))   # diagram (needs internet)
except Exception:
    print("(diagram not available)")

result = app.invoke({})

print("\n========== MACRO VIEW (AI) ==========")
print(result["macro"]["summary"])
print("\n========== PROPOSED ORDERS ==========")
for o in result["orders"]:
    print(f"- {o['action'].upper():4} {o['code']:<15} {o['amount_eur']:>10,.0f} EUR")
print("\n========== AFTER RISK CHECK ==========")
for o in result["risk"]["approved"]:
    note = f"  ({o['note']})" if o.get("note") else ""
    print(f"APPROVED  {o['action'].upper():4} {o['code']:<15} {o['amount_eur']:>10,.0f} EUR{note}")
for o in result["risk"]["rejected"]:
    print(f"REJECTED  {o['action'].upper():4} {o['code']:<15} -> {o['reason']}")
print(f"\nCash after trades: {result['risk']['cash_after_eur']:,.0f} EUR")
print("Weights after trades:", result["risk"]["weights_after"])

## 7. Still to decide / do

| # | What | With whom |
|---|------|-----------|
| 1 | Final limits (20% per ETF? leveraged cap 15%?) – justify with volatility/drawdown | Team |
| 2 | Order format keys (`code`, `action`, `amount_eur`) | Francesco |
| 3 | Connect the Portfolio Manager: it reads `macro["summary"]` and returns orders | Francesco + me |
| 4 | Optional: add macro numbers (ECB rate, VIX, oil) from FRED/ECB next to the news | Me |
| 5 | Optional extra rule: e.g. if VIX > 30, halve the leveraged cap | Team |